# Random Nucleotide Sampling

![Random Nucleotide Sampling](https://proto-bio.github.io/proto-assets/images/tool/random_nucleotide/hero.png)

This notebook demonstrates `run_random_nucleotide_sample`, which generates nucleotide sequence variants by filling masked positions with bases drawn from an IUPAC substitution scheme. It covers pre-masked sequences, automatic position selection via masking strategies, IUPAC scheme comparisons, RNA auto-detection, and batch library generation.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("random_nucleotide")
display_overview("random_nucleotide")
display_docs_section("random_nucleotide", "Background")

# Random Nucleotide Sampling

Random Nucleotide Sampling fills masked positions in DNA or RNA sequences with random bases drawn from an [IUPAC ambiguity code](https://en.wikipedia.org/wiki/Nucleic_acid_notation#IUPAC_notation) substitution pool. It supports configurable masking strategies, automatic DNA/RNA detection, and all 15 IUPAC degenerate base codes for controlling the substitution alphabet.

- **Tool key**: `random-nucleotide-sample`
- **Input**: DNA or RNA sequences (with optional `_` masks at positions to mutate)
- **Output**: Sequences with masked positions filled by random bases
- **Execution**: CPU only, no external dependencies

## Background

**What does this tool do?**
This tool performs random [mutagenesis](https://en.wikipedia.org/wiki/Mutagenesis) at the nucleotide level. Given a DNA or RNA sequence, it identifies positions to mutate (either pre-marked with `_` or selected by a masking strategy) and replaces them with random bases drawn from a specified substitution pool.

**Why is this important?**
Random mutagenesis is a foundational technique in:
- **[Directed evolution](https://en.wikipedia.org/wiki/Directed_evolution)**: Creating libraries of sequence variants for functional screening
- **Combinatorial library design**: Generating diversity at specific positions in promoters, UTRs, or coding regions
- **Robustness testing**: Evaluating how sensitive a designed sequence is to random perturbations
- **Degenerate codon libraries**: Using IUPAC codes to control which bases appear at each position

**IUPAC substitution schemes:**
The substitution scheme controls which bases can appear at masked positions:

| Code | Bases | Description |
|------|-------|-------------|
| `N` | A, C, G, T | Any base (maximum diversity) |
| `R` | A, G | Purines only |
| `Y` | C, T | Pyrimidines only |
| `S` | G, C | Strong (3 hydrogen bonds) |
| `W` | A, T | Weak (2 hydrogen bonds) |
| `K` | G, T | Keto |
| `M` | A, C | Amino |
| `B` | C, G, T | Not A |
| `D` | A, G, T | Not C |
| `H` | A, C, T | Not G |
| `V` | A, C, G | Not T |

## Available tools

In [2]:
display_available_tools("random_nucleotide")

- **`run_random_nucleotide_sample()`** — Sample nucleotide sequences by filling masked positions with random bases from an IUPAC substitution scheme

### `run_random_nucleotide_sample`

`run_random_nucleotide_sample` fills masked positions in DNA or RNA sequences with bases drawn from an IUPAC substitution scheme. Positions marked with `_` are replaced by a randomly sampled base; all other positions are preserved unchanged. The masking strategy system lets you automatically select which positions to diversify — by count, fraction, or specific indices — or you can pre-mask positions manually. The tool auto-detects DNA versus RNA by the presence of uracil and converts sampled thymine bases to uracil for RNA inputs.

In [3]:
from proto_tools import RandomNucleotideSampleInput, RandomNucleotideSampleConfig, run_random_nucleotide_sample
from proto_tools.transforms.masking import MaskingStrategy

In [4]:
# Display input docs
display_api_reference("random_nucleotide", "input", "run_random_nucleotide_sample")

# Underscore characters mark positions for random substitution
inputs = RandomNucleotideSampleInput(sequences=["ACGT_CGT_A"])

**Input** — `RandomNucleotideSampleInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | DNA/RNA sequence(s) to mutate. May contain '_' at positions to sample. |

In [5]:
# Display config docs
display_api_reference("random_nucleotide", "config", "run_random_nucleotide_sample")

# Use a masking strategy to automatically select 2 positions to mutate | see docs above for all fields
config = RandomNucleotideSampleConfig(
    masking_strategy=MaskingStrategy(num_mutations=2),
    seed=42,
)

**Config** — `RandomNucleotideSampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cpu'` | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `masking_strategy` | `proto_tools.transforms.masking.base.MaskingStrategy` | `MaskingStrategy(method='random', num_mutations=None, mask_fraction=None, fixed_positions=None, temperature=1.0, model_name=None, model_checkpoint=None)` | Controls which positions to mask for sampling. Default: random 30%. |
| `substitution_scheme` | `Literal['N', 'R', 'Y', 'S', 'W', 'K', 'M', 'B', 'D', 'H', 'V']` | `'N'` | IUPAC code defining the nucleotide substitution pool. |
| `sequence_type` | `Literal['auto', 'dna', 'rna']` | `'auto'` | Sequence type: auto-detect, force DNA, or force RNA. |

In [6]:
# Run the sampling function
wild_type = "ACGTACGTAC"
result = run_random_nucleotide_sample(
    RandomNucleotideSampleInput(sequences=[wild_type]),
    config,
)

In [7]:
# Display output docs
display_api_reference("random_nucleotide", "output", "run_random_nucleotide_sample")

# Show the variant and which positions were mutated
variant = result.sequences[0]
diffs = [i for i, (a, b) in enumerate(zip(wild_type, variant)) if a != b]
print(f"Wild type:         {wild_type}")
print(f"Variant:           {variant}")
print(f"Mutated positions: {diffs}")

# Demonstrate pre-masked input
result_premask = run_random_nucleotide_sample(RandomNucleotideSampleInput(sequences=["ACGT_CGT_A"]))
print(f"\nPre-masked input:  ACGT_CGT_A")
print(f"Variant:           {result_premask.sequences[0]}")

# Demonstrate IUPAC substitution schemes
print("\nIUPAC scheme comparison (all-masked sequence):")
for scheme in ["N", "R", "Y", "S", "W"]:
    scheme_config = RandomNucleotideSampleConfig(substitution_scheme=scheme, seed=42)
    scheme_result = run_random_nucleotide_sample(
        RandomNucleotideSampleInput(sequences=["____"]),
        scheme_config,
    )
    print(f"  {scheme}: {scheme_result.sequences[0]}")

# Demonstrate RNA auto-detection
rna_result = run_random_nucleotide_sample(RandomNucleotideSampleInput(sequences=["AUGC_CGU"]))
print(f"\nRNA input:    AUGC_CGU")
print(f"RNA variant:  {rna_result.sequences[0]}  (T converted to U)")

**Output** — `RandomNucleotideSampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | Nucleotide sequences with masked positions randomly filled |

Wild type:         ACGTACGTAC
Variant:           ACGAACGTAA
Mutated positions: [3, 9]

Pre-masked input:  ACGT_CGT_A
Variant:           ACGTACGTGA

IUPAC scheme comparison (all-masked sequence):
  N: AAGC
  R: AAGA
  Y: CCTC
  S: GGCG
  W: AATA

RNA input:    AUGC_CGU
RNA variant:  AUGCGCGU  (T converted to U)


## Export Results

Sampling results can be saved to FASTA, TXT, or JSON format for downstream analysis or synthesis ordering.

In [8]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Generate a batch of 5 variants from the same wild-type sequence
batch_inputs = RandomNucleotideSampleInput(sequences=["ACGTACGTAC"] * 5)
batch_config = RandomNucleotideSampleConfig(masking_strategy=MaskingStrategy(num_mutations=2))
batch_result = run_random_nucleotide_sample(batch_inputs, batch_config)

batch_result.export(name="random_nucleotide_variants", export_path=output_dir, file_format="fasta")
print(f"Exported sequences to {output_dir / 'random_nucleotide_variants.fasta'}")

Exported sequences to example_output/random_nucleotide_variants.fasta
